# LlamaIndex + CockroachDB walkthrough

This notebook walks through every public surface of `llama-index-cockroachdb`:

1. Connecting `CockroachDBVectorStore` to a CRDB v25.2+ cluster
2. Ingesting Documents with `VectorStoreIndex`
3. Default similarity search
4. MMR for diverse results
5. Metadata filters
6. `CockroachDBReader` against an existing CRDB table
7. `CockroachDBRetriever` with C-SPANN beam-size tuning

## Prerequisites

A running CockroachDB v25.2+ instance with the vector feature enabled. For a local single node:

```bash
cockroach start-single-node --insecure --listen-addr=localhost:26257 &
cockroach sql --insecure -e "SET CLUSTER SETTING feature.vector_index.enabled = true;"
```

## 1. Install

In [ ]:
# %pip install llama-index-cockroachdb llama-index-embeddings-openai llama-index-llms-openai

## 2. Connect to CockroachDB

`from_params` builds the right `cockroachdb+psycopg2://` and `cockroachdb+asyncpg://` URLs from individual fields and applies the C-SPANN partition kwargs at index creation time.

In [ ]:
from llama_index.vector_stores.cockroachdb import CockroachDBVectorStore

store = CockroachDBVectorStore.from_params(
    host="localhost",
    port=26257,
    user="root",
    password="",
    database="defaultdb",
    table_name="notebook_demo",
    embed_dim=1536,
    distance_metric="cosine",
    cspann_kwargs={"min_partition_size": 16, "max_partition_size": 128},
    sslmode="disable",  # local dev only
)

## 3. Ingest Documents

In [ ]:
import os

from llama_index.core import Document, StorageContext, VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

assert "OPENAI_API_KEY" in os.environ

docs = [
    Document(text="CockroachDB v25.2 ships a native VECTOR(n) column type."),
    Document(
        text="C-SPANN partitions the vector space so ANN search scales horizontally across nodes."
    ),
    Document(
        text="min_partition_size and max_partition_size control the tree shape at build time."
    ),
    Document(
        text="vector_search_beam_size is the runtime knob: higher beam = better recall, slightly more latency."
    ),
    Document(text="LlamaIndex retrievers compose with any BasePydanticVectorStore implementation."),
]

index = VectorStoreIndex.from_documents(
    docs,
    storage_context=StorageContext.from_defaults(vector_store=store),
    embed_model=OpenAIEmbedding(model="text-embedding-3-small"),
)

## 4. Default similarity search

In [ ]:
engine = index.as_query_engine(llm=OpenAI(model="gpt-4o-mini"), similarity_top_k=3)
print(engine.query("How do I tune CockroachDB vector search at runtime?"))

## 5. MMR for diverse top-k

`mmr_threshold` is the lever: 0 prioritizes diversity, 1 prioritizes relevance.

In [ ]:
from llama_index.core.vector_stores.types import VectorStoreQuery, VectorStoreQueryMode

embed = OpenAIEmbedding(model="text-embedding-3-small")
q_emb = embed.get_query_embedding("vector index tuning")

mmr = store.query(
    VectorStoreQuery(
        query_embedding=q_emb,
        similarity_top_k=3,
        mode=VectorStoreQueryMode.MMR,
        mmr_threshold=0.3,
    ),
    mmr_prefetch_factor=4.0,
)
for n, s in zip(mmr.nodes, mmr.similarities, strict=False):
    print(f"{s:.4f}  {n.get_content()[:80]}...")

## 6. Metadata filters

In [ ]:
from llama_index.core.vector_stores.types import (
    FilterCondition,
    FilterOperator,
    MetadataFilter,
    MetadataFilters,
)

filters = MetadataFilters(
    condition=FilterCondition.AND,
    filters=[
        MetadataFilter(key="year", value=2024, operator=FilterOperator.GTE),
    ],
)
# Pass through any query path that accepts filters; this example just shows the shape.

## 7. Reader: load existing CRDB rows as Documents

In [ ]:
from llama_index.readers.cockroachdb import CockroachDBReader

reader = CockroachDBReader.from_params(
    host="localhost",
    port=26257,
    database="defaultdb",
    user="root",
    password="",
    sslmode="disable",
)
# docs_from_db = reader.load_data(table="articles", text_column="body", metadata_columns=["id", "author"])
# Or with a parameterized query:
# docs_filtered = reader.load_data(
#     query="SELECT id, body, author FROM articles WHERE author = :a",
#     text_column="body",
#     metadata_columns=["id", "author"],
#     id_column="id",
#     params={"a": "alice"},
# )

## 8. Standalone retriever with beam-size tuning

In [ ]:
from llama_index.retrievers.cockroachdb import CockroachDBRetriever

retriever = CockroachDBRetriever(
    vector_store=store,
    embed_model=embed,
    similarity_top_k=3,
    vector_search_beam_size=128,
)
for n in retriever.retrieve("how does C-SPANN compare to HNSW"):
    print(f"{n.score:.4f}  {n.node.get_content()[:80]}...")

## 9. Cleanup

In [ ]:
store.clear()  # delete rows but keep the index
# import asyncio; asyncio.run(store.close())  # dispose engines